# Day 1: Setup + Data

**Goal**: Environment running, dataset downloaded, `images.csv` produced.

**Done checkpoint**:
- `data/metadata/images.csv` exists with clean rows
- `inspect_dataset()` prints stats without errors

**Runtime**: CPU is fine for Day 1.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Base project directory on your Drive
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Fast local disk for extracted data (lost on runtime restart, but much faster than Drive)
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Change working directory to project root for all scripts
%cd {PROJECT_DIR}
print(f'Working directory: {os.getcwd()}')

## 2. Clone Repo (first time only)

In [ ]:
import os

REPO_URL = 'https://github.com/atharvagasheTAMU/VisualSearchSystem.git'  # <-- update this

# If you haven't pushed to GitHub yet, manually copy the src/ folder to Drive instead.
# Once you have GitHub set up, this cell clones it once and skips if already cloned.
if not os.path.exists(os.path.join(PROJECT_DIR, 'src')):
    !git clone {REPO_URL} /tmp/repo
    !cp -r /tmp/repo/. {PROJECT_DIR}/
    print('Repo cloned.')
else:
    # Pull latest changes
    !git pull
    print('Repo already exists, pulled latest.')

## 3. Install Dependencies

In [ ]:
# Install all project dependencies
# This takes ~3-4 minutes on first run, much faster on subsequent runs
!pip install -q \
    torch torchvision \
    transformers \
    faiss-cpu \
    Pillow \
    pandas numpy tqdm \
    pyyaml python-dotenv \
    kaggle \
    fastapi uvicorn[standard] python-multipart aiofiles \
    streamlit \
    easyocr \
    wikipedia SPARQLWrapper \
    praw \
    httpx requests scikit-learn scipy
print('Installation complete.')

## 4. Configure API Credentials

In [ ]:
# --- Option A: Use Colab Secrets (recommended) ---
# Go to: Colab sidebar > Key icon > Add secrets
# Add: KAGGLE_USERNAME, KAGGLE_KEY

from google.colab import userdata
import os
import json
from pathlib import Path

try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    # Write kaggle.json
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(
        json.dumps({'username': kaggle_username, 'key': kaggle_key})
    )
    (kaggle_dir / 'kaggle.json').chmod(0o600)
    print('Kaggle credentials configured from Colab Secrets.')

except Exception:
    print('Colab Secrets not set. Using manual entry below.')

    # --- Option B: Manual entry ---
    # Uncomment and fill in if not using Secrets:
    # KAGGLE_USERNAME = 'your_username'
    # KAGGLE_KEY = 'your_api_key'
    # kaggle_dir = Path.home() / '.kaggle'
    # kaggle_dir.mkdir(exist_ok=True)
    # (kaggle_dir / 'kaggle.json').write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
    # (kaggle_dir / 'kaggle.json').chmod(0o600)
    pass

## 5. Download Datasets (Fashion + Landmarks + Food)

In [ ]:
import os, zipfile, shutil
from pathlib import Path

# Each dataset gets its own subfolder under raw/ for clean separation
# Drive: persistent zip storage   Local: fast extracted images for embedding

DATASETS = [
    {
        'name':       'fashion',
        'kaggle_slug': 'paramaggarwal/fashion-product-images-small',
        'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'fashion',
        'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'fashion',
        'check_glob': 'images/*.jpg',
        'size_hint':  '~570 MB',
    },
    {
        'name':       'landmarks',
        'kaggle_slug': 'vishnupriyavr/wiki-landmarks-dataset',
        'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'landmarks',
        'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'landmarks',
        'check_glob': '*/*.jpg',
        'size_hint':  '~300 MB',
    },
    {
        'name':       'food',
        'kaggle_slug': 'dansbecker/food-101',
        'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'food',
        'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'food',
        'check_glob': '*/*.jpg',
        'size_hint':  '~500 MB',
    },
]

def download_and_extract(ds):
    local_dir = ds['local_dir']
    drive_dir = ds['drive_dir']
    local_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)

    existing = list(local_dir.glob(ds['check_glob']))
    if existing:
        print(f"  [{ds['name']}] Already on local disk: {len(existing):,} images.")
        return

    # Try Drive zip first
    drive_zips = list(drive_dir.glob('*.zip'))
    if drive_zips:
        print(f"  [{ds['name']}] Extracting zip from Drive...")
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(local_dir)
        print(f"  [{ds['name']}] Done.")
        return

    # Download fresh from Kaggle
    print(f"  [{ds['name']}] Downloading {ds['size_hint']} from Kaggle...")
    os.system(f"kaggle datasets download -d {ds['kaggle_slug']} -p {local_dir}")

    local_zips = list(local_dir.glob('*.zip'))
    if not local_zips:
        print(f"  [{ds['name']}] No zip found — check Kaggle credentials.")
        return

    print(f"  [{ds['name']}] Saving zip to Drive for future sessions...")
    shutil.copy(local_zips[0], drive_dir)

    print(f"  [{ds['name']}] Extracting...")
    with zipfile.ZipFile(local_zips[0], 'r') as zf:
        zf.extractall(local_dir)
    print(f"  [{ds['name']}] Done.")

for ds in DATASETS:
    download_and_extract(ds)

print('\nAll datasets ready:')
for ds in DATASETS:
    count = len(list(ds['local_dir'].glob(ds['check_glob'])))
    print(f"  {ds['name']:12} {count:>7,} images   -> {ds['local_dir']}")

## 6. Inspect Datasets

In [ ]:
from pathlib import Path

# ── Fashion ──────────────────────────────────────────────────────────────────
fashion_dir = Path(LOCAL_DATA_DIR) / 'raw' / 'fashion'
styles_csv = next(fashion_dir.rglob('styles.csv'), None)
if styles_csv:
    import pandas as pd
    df_f = pd.read_csv(styles_csv, on_bad_lines='skip')
    print(f'[fashion] {df_f.shape[0]:,} rows')
    print(f'  masterCategory: {df_f["masterCategory"].value_counts().to_dict()}')
    print(f'  Top colors:     {df_f["baseColour"].value_counts().head(5).to_dict()}')
else:
    print('[fashion] styles.csv not found — run download cell first.')

# ── Landmarks ─────────────────────────────────────────────────────────────────
landmark_dir = Path(LOCAL_DATA_DIR) / 'raw' / 'landmarks'
if landmark_dir.exists():
    landmark_classes = [d.name for d in landmark_dir.iterdir() if d.is_dir()]
    print(f'\n[landmarks] {len(landmark_classes)} landmark classes')
    print(f'  Sample: {landmark_classes[:8]}')
else:
    print('\n[landmarks] not downloaded yet.')

# ── Food ──────────────────────────────────────────────────────────────────────
food_dir = Path(LOCAL_DATA_DIR) / 'raw' / 'food'
if food_dir.exists():
    food_classes = [d.name for d in food_dir.iterdir() if d.is_dir()]
    print(f'\n[food] {len(food_classes)} food classes')
    print(f'  Sample: {food_classes[:8]}')
else:
    print('\n[food] not downloaded yet.')

## 7. Clean All Datasets → images.csv

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, PROJECT_DIR)

import yaml
with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

# Point each dataset's raw_dir to the local fast disk
for ds in config['datasets']:
    ds['raw_dir'] = str(LOCAL / 'raw' / ds['name'])

# Outputs saved to Drive (persistent)
config['paths']['metadata_csv'] = str(BASE / 'data' / 'metadata' / 'images.csv')
config['paths']['metadata_dir'] = str(BASE / 'data' / 'metadata')

from src.preprocessing.clean import clean_all_datasets
df = clean_all_datasets(config)
df.head(10)

## ✅ Day 1 Done Checkpoint

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(PROJECT_DIR) / 'data' / 'metadata' / 'images.csv'
assert csv_path.exists(), 'images.csv not found!'
df = pd.read_csv(csv_path)
assert len(df) > 0, 'images.csv is empty!'
assert 'dataset' in df.columns, 'dataset column missing — re-run clean cell'
assert 'article_type' in df.columns, 'article_type column missing'

print('Day 1 COMPLETE')
print(f'  images.csv: {len(df):,} rows')
print(f'  Columns:    {list(df.columns)}')
print(f'\n  Per dataset:')
for ds_name, grp in df.groupby('dataset'):
    cats = grp['category'].nunique()
    types = grp['article_type'].nunique() if 'article_type' in grp else '?'
    print(f'    {ds_name:12} {len(grp):>7,} images  |  {cats} categories  |  {types} article types')